In [1]:
import pandas as pd

In [ ]:
xl_path = r"../data/wwtp_scenarios_summary.xlsx"
df = pd.read_excel(xl_path)
df["vassom"] = df["regine"].str.split(".", expand=True)[0].astype(int)
df = df.query("5 <= vassom <= 9")
df.head()

In [ ]:
par = "totn"
scen_dict = {"Baseline": 2, "Scenario_A": 2, "Scenario_B": 1}
df["overflow_pct"] = df["scenario"].map(scen_dict)
df["overflow_tonnes"] = df[f"{par}_in_tonnes"] * df["overflow_pct"] / 100
df[f"eff_{par}"] = (
    100
    * (df[f"{par}_in_tonnes"] - (df[f"{par}_out_tonnes"] - df["overflow_tonnes"]))
    / df[f"{par}_in_tonnes"]
)
df.head()

In [ ]:
# df.to_excel('test.xlsx', index=False)

In [ ]:
df.groupby(["anlegg_nr", "scenario"]).agg(
    {"anlegg_name": "first", "current_capacity": "mean", f"eff_{par}": "mean"}
)

In [25]:
xl_path = r"../data/wwtp_scenarios_summary.xlsx"
df = pd.read_excel(xl_path)
df["vassom"] = df["regine"].str.split(".", expand=True)[0].astype(int)
df = df.query("5 <= vassom <= 9")

scen_dict = {"Baseline": 2, "Scenario_A": 2, "Scenario_B": 1}
df["overflow_pct"] = df["scenario"].map(scen_dict)

bins = [0, 1000, 5000, 10000, float("inf")]
labels = ["<1k p.e.", "1k to 5k p.e.", "5k to 10k p.e.", ">10k p.e."]
df["cap_class"] = pd.cut(df["current_capacity"], bins=bins, labels=labels, right=False)

cols = []
for par in ["totn", "totp"]:
    df[f"{par}_overflow_tonnes"] = df[f"{par}_in_tonnes"] * df["overflow_pct"] / 100
    df[f"{par}_out_tonnes"] = df[f"{par}_out_tonnes"] - df[f"{par}_overflow_tonnes"]
    cols.append(f"{par}_out_tonnes")
    cols.append(f"{par}_overflow_tonnes")

id_cols = ["scenario", "cap_class"]
df = df[id_cols + cols]
df = df.groupby(id_cols, observed=False).sum().reset_index()

df

,scenario,cap_class,totn_out_tonnes,totn_overflow_tonnes,totp_out_tonnes,totp_overflow_tonnes
0,Baseline,<1k p.e.,18.69376,0.51124,0.11958,0.05442
1,Baseline,1k to 5k p.e.,48.62342,1.21558,0.68272,0.15528
2,Baseline,5k to 10k p.e.,129.82142,3.24558,0.76104,0.27996
3,Baseline,>10k p.e.,4625.99990,279.13310,140.97666,31.93434
4,Scenario_A,<1k p.e.,18.69376,0.51124,0.11958,0.05442
5,Scenario_A,1k to 5k p.e.,48.62342,1.21558,0.68272,0.15528
6,Scenario_A,5k to 10k p.e.,129.82142,3.24558,0.76104,0.27996
7,Scenario_A,>10k p.e.,2791.33190,279.13310,140.97666,31.93434
8,Scenario_B,<1k p.e.,18.94938,0.25562,0.14679,0.02721
9,Scenario_B,1k to 5k p.e.,48.62221,0.60779,0.68336,0.07764


In [26]:
base_df = df.query("scenario == 'Baseline'").sort_values(id_cols).copy()
df_list = []
for scen in ("Scenario_A", "Scenario_B"):
    scen_df = df.query("scenario == @scen").sort_values(id_cols).copy()
    for col in cols:
        scen_df[col] = (
            100 * (scen_df[col].values - base_df[col].values) / base_df[col].values
        )
    df_list.append(scen_df)
df = pd.concat(df_list, axis="rows")
df

,scenario,cap_class,totn_out_tonnes,totn_overflow_tonnes,totp_out_tonnes,totp_overflow_tonnes
4,Scenario_A,<1k p.e.,0.000000,0.0,0.000000,0.0
5,Scenario_A,1k to 5k p.e.,0.000000,0.0,0.000000,0.0
6,Scenario_A,5k to 10k p.e.,0.000000,0.0,0.000000,0.0
7,Scenario_A,>10k p.e.,-39.659923,0.0,0.000000,0.0
8,Scenario_B,<1k p.e.,1.367408,-50.0,22.754641,-50.0
9,Scenario_B,1k to 5k p.e.,-0.002489,-50.0,0.093743,-50.0
10,Scenario_B,5k to 10k p.e.,-62.499863,-50.0,-0.002628,-50.0
11,Scenario_B,>10k p.e.,-54.713911,-50.0,0.000830,-50.0


In [21]:
res_df = (
    pd.read_csv(r"../data/results_summary.csv")
    .query(
        "(`Område` == 'Indre Oslofjord') and (Kilde != 'Bakgrunn') and (Scenario == 'Baseline')"
    )
    .drop(columns=["Scenario", "Område", "Kilde"])
    .groupby("Parameter")
    .sum()
)
res_df

,Verdi (tonn)
Parameter,
DIN,1737.088648
SS,15199.767569
TDP,43.063686
TOC,3568.342830
TON,621.717027
TOTN,2358.805674
TOTP,93.764625
TPP,50.700939


In [40]:
xl_path = r"../data/wwtp_scenarios_summary.xlsx"
df = pd.read_excel(xl_path)
df["vassom"] = df["regine"].str.split(".", expand=True)[0].astype(int)

# bins = [0, 1000, 5000, 10000, float("inf")]
# labels = ["<1k p.e.", "1k to 5k p.e.", "5k to 10k p.e.", ">10k p.e."]
# df["cap_class"] = pd.cut(df["current_capacity"], bins=bins, labels=labels, right=False)

pars = ["totn", "totp", "bof5", "ss"]

cols = [
    "scenario",
    "anlegg_nr",
    "anlegg_name",
    "outlet_zone",
    "outlet_east",
    "outlet_north",
    'vassom',
]
val_cols = [f"{par}_out_tonnes" for par in pars]

agg_dict = {
    "anlegg_name": "first",
    "outlet_zone": "first",
    "outlet_east": "mean",
    "outlet_north": "mean",
    'vassom':'mean',
}
for col in val_cols:
    agg_dict[col] = "sum"

df = df[cols + val_cols].groupby(["scenario", "anlegg_nr"]).agg(agg_dict).reset_index()

base_df = df.query("scenario == 'Baseline'").copy()
df_list = []
for scen in ("Scenario_A", "Scenario_B"):
    scen_df = df.query("scenario == @scen").copy()
    scen_df = pd.merge(
        scen_df,
        base_df[["anlegg_nr"] + val_cols],
        how="outer",
        on="anlegg_nr",
        suffixes=("", "_base"),
    )
    for par in pars:
        scen_df[f"{par}_reduct_pct"] = (
            100
            * (scen_df[f"{par}_out_tonnes_base"] - scen_df[f"{par}_out_tonnes"])
            / scen_df[f"{par}_out_tonnes_base"]
        )
        del scen_df[f"{par}_out_tonnes_base"], scen_df[f"{par}_out_tonnes"]
    df_list.append(scen_df)
df = pd.concat(df_list, axis='rows').fillna(0)
df.to_excel('test.xlsx', index=False)